# 02 - Covariance front end on the FPGA (Tier 1)

`scm_unitary` IP: unitary transform + real-valued sample covariance, fixed point.

1. Bit-exact check of the board output against the integer model
2. DOA from the hardware covariance (MUSIC on the ARM), uncorrelated and coherent sources
3. Host-observed timing: FPGA vs NumPy on the ARM

Upload `scm_overlay.bit` and `scm_overlay.hwh` into the same folder as this notebook.

In [ ]:
import os, time
import numpy as np
import pynq
from pynq import Overlay, allocate
print("PYNQ", pynq.__version__, "| numpy", np.__version__)

ol = Overlay(os.path.abspath("scm_overlay.bit"))
print("IP blocks:", list(ol.ip_dict.keys()))
dma = ol.axi_dma_0
scm_ip = ol.scm_unitary_0

N, M = 8, 128
IN_FRAC, OUT_FRAC = 15, 27

In [ ]:
# ---- Bit-exact integer model of the IP (same as python/scm_bitexact.py) ----
def quantize_input(x):
    q = lambda v: np.clip(np.floor(v * 2**IN_FRAC + 0.5), -2**15, 2**15 - 1).astype(np.int64)
    return q(x.real), q(x.imag)

def scm_model_raw(xr, xi):
    h = N // 2
    a = xr[:h] - xr[::-1][:h]
    b = xi[:h] - xi[::-1][:h]
    yr = np.vstack([xr[:h] + xr[::-1][:h], b])
    yi = np.vstack([xi[:h] + xi[::-1][:h], -a])
    s = yr @ yr.T + yi @ yi.T
    shift = 2 * IN_FRAC + 8 - OUT_FRAC
    return np.clip((s + (1 << (shift - 1))) >> shift, -2**31, 2**31 - 1)

def pack(xr, xi):
    # one complex sample per word, [15:0] re, [31:16] im, snapshot-major
    return (xr.T.astype(np.uint16).astype(np.uint32)
            | (xi.T.astype(np.uint16).astype(np.uint32) << 16)).ravel()

in_buf = allocate(shape=(N * M,), dtype=np.uint32)
out_buf = allocate(shape=(N * N,), dtype=np.uint32)

def scm_fpga_raw(words):
    in_buf[:] = words
    dma.recvchannel.transfer(out_buf)
    scm_ip.write(0x00, 0x01)              # ap_start
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    return np.array(out_buf).view(np.int32).astype(np.int64).reshape(N, N)

def normalize(x):
    return x / (1.001 * max(np.abs(x.real).max(), np.abs(x.imag).max(), 1e-12))

In [ ]:
# ---- Test 1: bit-exact on 50 random frames ----
rng = np.random.RandomState(1)
fails = 0
for t in range(50):
    x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
    xr, xi = quantize_input(x)
    hw = scm_fpga_raw(pack(xr, xi))
    ref = scm_model_raw(xr, xi)
    if not np.array_equal(hw, ref):
        fails += 1
        if fails <= 3:
            print("frame", t, "max |diff| =", np.abs(hw - ref).max())
print("bit-exact frames: %d / 50 %s" % (50 - fails, "PASS" if fails == 0 else "FAIL"))

In [ ]:
# ---- Test 2: DOA from the hardware covariance (unitary MUSIC on the ARM) ----
pos = (np.arange(N) - (N - 1) / 2) * 0.5          # centred virtual ULA, d = lambda/2
grid = np.arange(-90.0, 90.5, 0.5)

def steering(theta_deg):
    return np.exp(1j * 2 * np.pi * np.outer(pos, np.sin(np.deg2rad(np.atleast_1d(theta_deg)))))

h = N // 2
I, J = np.eye(h), np.fliplr(np.eye(h))
Q = np.block([[I, 1j * I], [J, -1j * J]]) / np.sqrt(2)
B = (Q.conj().T @ steering(grid)).real               # real steering table

def simulate(theta, snr_db, coherent=False):
    k = len(theta)
    s = (rng.randn(k, M) + 1j * rng.randn(k, M)) / np.sqrt(2)
    if coherent:
        s = np.repeat(s[:1], k, axis=0) * np.exp(1j * rng.uniform(0, 2 * np.pi, (k, 1)))
    n = 10 ** (-snr_db / 20) * (rng.randn(N, M) + 1j * rng.randn(N, M)) / np.sqrt(2)
    return steering(theta) @ s + n

def music_peaks(R, k):
    w, v = np.linalg.eigh(R)
    Es = v[:, -k:]
    d = np.sum(B**2, axis=0) - np.sum((Es.T @ B)**2, axis=0)
    p = 10 * np.log10(1.0 / np.maximum(d, 1e-12))
    mins = [i for i in range(1, len(d) - 1) if d[i] < d[i-1] and d[i] <= d[i+1]]
    mins = sorted(mins, key=lambda i: d[i])[:k]
    return np.sort(grid[mins]), p

true = np.array([-10.0, 5.0])
results = {}
for coherent in (False, True):
    x = normalize(simulate(true, 10, coherent))
    xr, xi = quantize_input(x)
    R = scm_fpga_raw(pack(xr, xi)) * 2.0**-OUT_FRAC
    est, spec = music_peaks(R, 2)
    results[coherent] = spec
    print("%-12s true %s  ->  estimated %s deg" % ("coherent:" if coherent else "uncorrelated:", true, est))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for coherent, spec in results.items():
    plt.plot(grid, spec - spec.max(), label="coherent" if coherent else "uncorrelated")
for t in true:
    plt.axvline(t, color="gray", ls=":")
plt.xlabel("angle (deg)"); plt.ylabel("MUSIC spectrum (dB)")
plt.title("Unitary MUSIC from the FPGA covariance, N=8, M=128, SNR 10 dB")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ---- Test 3: host-observed time, FPGA vs NumPy on the ARM (median of 200) ----
x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
xr, xi = quantize_input(x)
words = pack(xr, xi)
xq = (xr + 1j * xi) * 2.0**-IN_FRAC

def median_us(fn, reps=200):
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return 1e6 * np.median(t)

def numpy_scm():
    y = Q.conj().T @ xq
    return (y @ y.conj().T).real / M

t_fpga = median_us(lambda: scm_fpga_raw(words))
t_np_float = median_us(numpy_scm)
t_np_exact = median_us(lambda: scm_model_raw(xr, xi))
print("FPGA (DMA + IP, host-observed): %8.1f us" % t_fpga)
print("NumPy float, ARM              : %8.1f us" % t_np_float)
print("NumPy integer model, ARM      : %8.1f us" % t_np_exact)
print("HLS estimate of the IP alone  : ~11 us at 100 MHz (see hls/scm/reports)")